# Análise de sinistros nas rodovias federais do Brasil

Este notebook reúne a lógica do projeto em uma sequência única e documentada. Cada etapa explica o que faz, quais arquivos utiliza e quais resultados produz.

## Fluxo do processamento

1. Baixar os dados públicos da Polícia Rodoviária Federal.
2. Consolidar os arquivos anuais de ocorrência e pessoa/veículo.
3. Normalizar marca, modelo e família dos veículos.
4. Enriquecer os registros com resultados do Latin NCAP.
5. Enriquecer os registros com a base local da FIPE.
6. Filtrar veículos leves com pareamento Latin NCAP.
7. Consolidar a base analítica e gerar o resumo de cobertura.

A célula final executa o fluxo completo com retomada e gera o CSV analítico consolidado.


## Preparação do ambiente

As funções abaixo carregam os módulos incorporados nas próximas seções. O código-fonte de cada script está armazenado no próprio notebook para que ele possa ser lido e executado sem depender da chamada de outros scripts por `subprocess`.


In [ ]:
from pathlib import Path
import sys
import types

RAIZ_PROJETO = Path.cwd()
if not (RAIZ_PROJETO / "scripts").exists() and (RAIZ_PROJETO.parent / "scripts").exists():
    RAIZ_PROJETO = RAIZ_PROJETO.parent

if str(RAIZ_PROJETO / "scripts") not in sys.path:
    sys.path.insert(0, str(RAIZ_PROJETO / "scripts"))

modulos = {}
print(f"Pasta do projeto: {RAIZ_PROJETO}")


### download_prf_data.py

Baixa e extrai os arquivos anuais dos dados abertos da PRF, registrando um manifesto.

In [ ]:
modulo_download_prf_data = types.ModuleType('notebook_download_prf_data')
sys.modules['notebook_download_prf_data'] = modulo_download_prf_data
namespace_download_prf_data = modulo_download_prf_data.__dict__
exec(compile(r'''
#!/usr/bin/env python3

from __future__ import annotations

import argparse
import csv
import html
import re
import subprocess
import sys
import urllib.error
import urllib.parse
import urllib.request
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable


PRF_OPEN_DATA_URL = (
    "https://www.gov.br/prf/pt-br/acesso-a-informacao/"
    "dados-abertos/dados-abertos-da-prf"
)
USER_AGENT = "prf-analise-downloader/1.0"
DEFAULT_KINDS = ("ocorrencia", "pessoa")
MANIFEST_FIELDS = [
    "year",
    "kind",
    "label",
    "source_url",
    "file_id",
    "download_url",
    "local_zip",
    "extract_dir",
    "extracted_files",
    "status",
    "bytes",
]


@dataclass(frozen=True)
class PrfDownload:
    year: int
    kind: str
    label: str
    source_url: str
    file_id: str

    @property
    def download_url(self) -> str:
        return f"https://drive.google.com/uc?export=download&id={self.file_id}"


def request_url(url: str, timeout: int = 90) -> urllib.request.Request:
    return urllib.request.Request(url, headers={"User-Agent": USER_AGENT})


def fetch_html(url: str) -> str:
    with urllib.request.urlopen(request_url(url), timeout=90) as response:
        return response.read().decode("utf-8", errors="replace")


def strip_tags(value: str) -> str:
    value = re.sub(r"<[^>]+>", " ", value)
    return " ".join(html.unescape(value).replace("\xa0", " ").split())


def drive_file_id(url: str) -> str:
    match = re.search(r"/file/d/([^/]+)/", url)
    if match:
        return match.group(1)
    parsed = urllib.parse.urlparse(url)
    query = urllib.parse.parse_qs(parsed.query)
    return query.get("id", [""])[0]


def row_kind(label: str) -> str:
    normalized = label.lower()
    if "todas as causas" in normalized:
        return "pessoa_todas_causas"
    if "agrupados por ocorrência" in normalized or "agrupados por ocorrencia" in normalized:
        return "ocorrencia"
    if "agrupados por pessoa" in normalized:
        return "pessoa"
    return ""


def visible_download_href(cell_html: str) -> str:
    anchors = re.findall(
        r"<a\b(?P<attrs>[^>]*)>(?P<body>.*?)</a>",
        cell_html,
        flags=re.IGNORECASE | re.DOTALL,
    )
    fallback = ""
    for attrs, body in anchors:
        href_match = re.search(r'href="([^"]+)"', attrs)
        if not href_match:
            continue
        href = html.unescape(href_match.group(1))
        fallback = fallback or href
        if "baixar" in strip_tags(body).lower():
            return href
    return fallback


def parse_accident_links(page_html: str) -> list[PrfDownload]:
    rows: list[PrfDownload] = []
    for match in re.finditer(
        r"<tr>\s*<td>(?P<label>Documento CSV de Acidentes .*?)</td>\s*"
        r"<td>(?P<link_cell>.*?)</td>\s*</tr>",
        page_html,
        flags=re.IGNORECASE | re.DOTALL,
    ):
        label = strip_tags(match.group("label"))
        year_match = re.search(r"\b(20\d{2})\b", label)
        kind = row_kind(label)
        href = visible_download_href(match.group("link_cell"))
        file_id = drive_file_id(href)
        if not year_match or not kind or not href or not file_id:
            continue
        rows.append(
            PrfDownload(
                year=int(year_match.group(1)),
                kind=kind,
                label=label,
                source_url=href,
                file_id=file_id,
            )
        )
    rows.sort(key=lambda row: (row.year, row.kind))
    return rows


def selected_links(
    links: Iterable[PrfDownload],
    start_year: int,
    end_year: int,
    kinds: set[str],
) -> list[PrfDownload]:
    selected = [
        link
        for link in links
        if start_year <= link.year <= end_year and link.kind in kinds
    ]
    selected.sort(key=lambda row: (row.year, row.kind))
    return selected


def download_file(url: str, output_path: Path, overwrite: bool) -> int:
    if output_path.exists() and not overwrite:
        return output_path.stat().st_size

    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = output_path.with_suffix(output_path.suffix + ".part")
    if temporary_path.exists():
        temporary_path.unlink()
    subprocess.run(
        [
            "curl",
            "-L",
            "--fail",
            "--retry",
            "3",
            "--connect-timeout",
            "30",
            "--max-time",
            "300",
            "--silent",
            "--show-error",
            "-o",
            str(temporary_path),
            url,
        ],
        check=True,
    )
    temporary_path.replace(output_path)
    return output_path.stat().st_size


def extract_zip(zip_path: Path, extract_dir: Path, overwrite: bool) -> list[str]:
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        names = [name for name in archive.namelist() if not name.endswith("/")]
        for name in names:
            target = extract_dir / name
            if target.exists() and not overwrite:
                continue
            archive.extract(name, extract_dir)
    return names


def write_manifest(path: Path, rows: list[dict[str, object]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=MANIFEST_FIELDS)
        writer.writeheader()
        writer.writerows(rows)


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--source-url", default=PRF_OPEN_DATA_URL)
    parser.add_argument("--start-year", type=int, default=2010)
    parser.add_argument("--end-year", type=int, default=2026)
    parser.add_argument(
        "--kinds",
        nargs="+",
        choices=("ocorrencia", "pessoa", "pessoa_todas_causas"),
        default=list(DEFAULT_KINDS),
    )
    parser.add_argument("--output-dir", type=Path, default=Path("data/raw/prf/acidentes"))
    parser.add_argument(
        "--manifest-output",
        type=Path,
        default=Path("data/raw/prf/manifest_acidentes_2010_2026.csv"),
    )
    parser.add_argument("--dry-run", action="store_true")
    parser.add_argument("--overwrite", action="store_true")
    parser.add_argument("--limit", type=int, default=0)
    return parser


def main() -> None:
    args = build_parser().parse_args()
    links = selected_links(
        parse_accident_links(fetch_html(args.source_url)),
        args.start_year,
        args.end_year,
        set(args.kinds),
    )
    if args.limit:
        links = links[: args.limit]

    manifest_rows: list[dict[str, object]] = []
    for position, link in enumerate(links, start=1):
        zip_path = args.output_dir / link.kind / f"{link.year}.zip"
        extract_dir = args.output_dir / link.kind / str(link.year)
        row = {
            "year": link.year,
            "kind": link.kind,
            "label": link.label,
            "source_url": link.source_url,
            "file_id": link.file_id,
            "download_url": link.download_url,
            "local_zip": str(zip_path),
            "extract_dir": str(extract_dir),
            "extracted_files": "",
            "status": "dry_run" if args.dry_run else "pending",
            "bytes": "",
        }
        if not args.dry_run:
            try:
                print(
                    f"[{position}/{len(links)}] Baixando {link.year} {link.kind}",
                    flush=True,
                )
                size = download_file(link.download_url, zip_path, overwrite=args.overwrite)
                extracted = extract_zip(zip_path, extract_dir, overwrite=args.overwrite)
                row.update(
                    {
                        "extracted_files": "|".join(extracted),
                        "status": "ok",
                        "bytes": size,
                    }
                )
            except (
                OSError,
                subprocess.CalledProcessError,
                zipfile.BadZipFile,
                urllib.error.URLError,
            ) as exc:
                row["status"] = f"erro: {exc}"
                print(f"Erro em {link.year} {link.kind}: {exc}", file=sys.stderr)
        manifest_rows.append(row)

    write_manifest(args.manifest_output, manifest_rows)
    print(f"Links selecionados: {len(links)}")
    print(f"Manifesto: {args.manifest_output}")


''', 'download_prf_data.py', 'exec'), namespace_download_prf_data)
modulos['download_prf_data'] = namespace_download_prf_data
print('Código incorporado: download_prf_data.py')
